In [ ]:
import os, glob, time
import numpy as np, pandas as pd, torch, soundfile as sf
from transformers import AutoFeatureExtractor, AutoModel

WORK = '/kaggle/working'
def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
folds = pd.read_csv(find('folds.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
uids = list(folds.uid) + ['test/' + f for f in test.filename]
for u in (uids[0], uids[-1]):
    assert os.path.exists(f'{BASE}/{u}'), f'audio path not found: {BASE}/{u} | contents: {os.listdir(BASE)}'
pd.DataFrame({'uid': uids}).to_csv(f'{WORK}/enc_uids.csv', index=False)

dev = 'cuda'
MODELS = {'hubert': 'facebook/hubert-large-ll60k', 'w2v2': 'facebook/wav2vec2-large-lv60'}
for tag, name in MODELS.items():
    fe = AutoFeatureExtractor.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(dev).half().eval()
    means, stds, t0 = [], [], time.time()
    for i, u in enumerate(uids):
        wav, sr = sf.read(f'{BASE}/{u}', dtype='float32')
        assert sr == 16000
        if wav.ndim > 1:
            wav = wav.mean(1)
        x = fe(wav, sampling_rate=16000, return_tensors='pt').input_values.to(dev).half()
        with torch.inference_mode():
            hs = torch.stack(model(x, output_hidden_states=True).hidden_states)[:, 0].float()
        means.append(hs.mean(1).cpu().numpy().astype(np.float16))
        stds.append(hs.std(1).cpu().numpy().astype(np.float16))
        if (i + 1) % 200 == 0:
            print(tag, f'{i+1}/{len(uids)} | {(time.time()-t0)/60:.1f} min')
    M, S = np.stack(means), np.stack(stds)
    print(tag, M.shape, '| NaN:', bool(np.isnan(M.astype(np.float32)).any()))
    np.save(f'{WORK}/{tag}_mean.npy', M)
    np.save(f'{WORK}/{tag}_std.npy', S)
    del model; torch.cuda.empty_cache()